# 作业 4：正则表达式与结构化网页数据

**学号**：20244080134　**姓名**：平吉佳

In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080134

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境 ] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[ 学号 ]       {STUDENT_ID}")
print(f"[ 机器指纹 ] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境 ] exp_env
[ 学号 ]       20244080134
[ 机器指纹 ] bdba4bb6344f9593
[RUN_ID]   20244080134-bdba4bb6344f9593


## 1 Python 正则表达式练习

以下 12 道题每题编写一个 Python 函数，并用 `check()` 展示测试用例。

In [2]:
import re


def check(actual, expected):
    """测试辅助函数：比较实际值与期望值，打印 PASS/FAIL。"""
    ok = actual == expected
    print(("PASS" if ok else "FAIL") + f": {actual!r}" + ("" if ok else f"  (期望 {expected!r})"))
    return ok

### 1.1 检查字符串是否仅包含英文字母 a-z、A-Z 和数字 0-9

用 `fullmatch` 要求**整个字符串**由 `[A-Za-z0-9]` 组成；用 `+` 表示至少一个字符（空串按不满足处理）。

In [3]:
def is_alnum_only(s):
    """检查字符串是否仅由英文字母 a-z、A-Z 和数字 0-9 组成（至少一个字符）。"""
    return re.fullmatch(r"[A-Za-z0-9]+", s) is not None


check(is_alnum_only("abc123"), True)
check(is_alnum_only("HelloWorld99"), True)
check(is_alnum_only("abc-123"), False)     # 含连字符
check(is_alnum_only("abc 123"), False)     # 含空格
check(is_alnum_only("你好abc"), False)     # 含中文
check(is_alnum_only(""), False)            # 空串：按不满足处理

PASS: True
PASS: True
PASS: False
PASS: False
PASS: False
PASS: False


True

### 1.2 匹配一个 a 后面跟着零个或多个 b 的字符串

`ab*`：`a` 之后 `b*` 匹配零个或多个 `b`。

In [4]:
def match_a_then_zero_or_more_b(s):
    """整个字符串为 'a' 后跟零个或多个 'b' 时返回 True。"""
    return re.fullmatch(r"ab*", s) is not None


check(match_a_then_zero_or_more_b("a"), True)
check(match_a_then_zero_or_more_b("ab"), True)
check(match_a_then_zero_or_more_b("abb"), True)
check(match_a_then_zero_or_more_b("abbb"), True)
check(match_a_then_zero_or_more_b("b"), False)      # 没有 a
check(match_a_then_zero_or_more_b("ac"), False)     # b 的位置出现 c
check(match_a_then_zero_or_more_b("ba"), False)     # a 不在开头
check(match_a_then_zero_or_more_b("abab"), False)   # 多出一段 ab

PASS: True
PASS: True
PASS: True
PASS: True
PASS: False
PASS: False
PASS: False
PASS: False


True

### 1.3 匹配一个 a 后面跟着一个或多个 b 的字符串

`ab+`：`b+` 匹配一个或多个 `b`（与 1.2 的区别仅在于 `"a"` 不再匹配）。

In [5]:
def match_a_then_one_or_more_b(s):
    """整个字符串为 'a' 后跟一个或多个 'b' 时返回 True。"""
    return re.fullmatch(r"ab+", s) is not None


check(match_a_then_one_or_more_b("ab"), True)
check(match_a_then_one_or_more_b("abb"), True)
check(match_a_then_one_or_more_b("abbb"), True)
check(match_a_then_one_or_more_b("a"), False)       # 至少需要一个 b
check(match_a_then_one_or_more_b("b"), False)
check(match_a_then_one_or_more_b("abab"), False)

PASS: True
PASS: True
PASS: True
PASS: False
PASS: False
PASS: False


True

### 1.4 匹配以 a 开头、以 b 结尾、中间任意字符的字符串

`a.*b`：`.` 匹配任意字符，`*` 表示零个或多个；加 `re.DOTALL` 使 `.` 也匹配换行符。

In [6]:
def match_a_any_b(s):
    """字符串以 'a' 开头、以 'b' 结尾、中间为任意字符（含换行）时返回 True。"""
    return re.fullmatch(r"a.*b", s, re.DOTALL) is not None


check(match_a_any_b("ab"), True)          # 中间为空也匹配
check(match_a_any_b("a123b"), True)
check(match_a_any_b("axxbyyb"), True)
check(match_a_any_b("a\nb"), True)       # re.DOTALL 使 . 匹配换行符
check(match_a_any_b("ac"), False)         # 不以 b 结尾
check(match_a_any_b("ba"), False)         # 不以 a 开头
check(match_a_any_b("abx"), False)        # 结尾多了 x

PASS: True
PASS: True
PASS: True
PASS: True
PASS: False
PASS: False
PASS: False


True

### 1.5 删除 IP 地址各段数字的前导零

要点：`\b0+(?=\d)` 在**单词边界**处匹配一个或多个 `0`，且后面必须还跟着数字（前瞻断言不消耗字符）：
- `100` 中间的 `0` 前是数字，不构成单词边界，不会被删；
- 全零段 `000` 会删掉前两个 `0` 而保留最后一个（之后没有数字，前瞻失败），即保留一个 `0`。

In [7]:
def remove_leading_zeros_in_ip(ip):
    """删除 IP 地址各段的前导零；某段全为零时保留一个 0。"""
    return re.sub(r"\b0+(?=\d)", "", ip)


check(remove_leading_zeros_in_ip("192.168.001.010"), "192.168.1.10")   # 题目示例
check(remove_leading_zeros_in_ip("001.002.003.004"), "1.2.3.4")
check(remove_leading_zeros_in_ip("192.168.000.000"), "192.168.0.0")    # 全零段保留一个 0
check(remove_leading_zeros_in_ip("0.0.0.0"), "0.0.0.0")                # 全零段不变
check(remove_leading_zeros_in_ip("10.0.0.1"), "10.0.0.1")              # 本来无前导零
check(remove_leading_zeros_in_ip("100.200.0.00"), "100.200.0.0")       # 段内 0 不受影响

PASS: '192.168.1.10'
PASS: '1.2.3.4'
PASS: '192.168.0.0'
PASS: '0.0.0.0'
PASS: '10.0.0.1'
PASS: '100.200.0.0'


True

### 1.6 将空格替换为下划线，同时将下划线替换为空格

不能简单地先 `replace(' ', '_')` 再 `replace('_', ' ')`，第二步会把刚生成的下划线又换回去。用 `re.sub` 配合回调函数，对每个匹配到的空格/下划线按原字符一次性互换。

In [8]:
def swap_space_underscore(s):
    """空格 <-> 下划线 互换（一次扫描完成，避免二次替换互相干扰）。"""
    return re.sub(r"[ _]", lambda m: "_" if m.group(0) == " " else " ", s)


check(swap_space_underscore("a b_c d"), "a_b c_d")
check(swap_space_underscore("hello_world again"), "hello world_again")
check(swap_space_underscore(" hello_world "), "_hello world_")
check(swap_space_underscore("a_b"), "a b")

PASS: 'a_b c_d'
PASS: 'hello world_again'
PASS: '_hello world_'
PASS: 'a b'


True

### 1.7 将 yyyy-mm-dd 格式日期转换为 dd-mm-yyyy 格式

用三个捕获组分别记住年、月、日，替换时倒序引用 `\3-\2-\1`；字符串中的其他内容保持不变。

In [9]:
def yyyy_mm_dd_to_dd_mm_yyyy(s):
    """把字符串中的 yyyy-mm-dd 日期转换为 dd-mm-yyyy。"""
    return re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3-\2-\1", s)


check(yyyy_mm_dd_to_dd_mm_yyyy("2024-03-15"), "15-03-2024")
check(yyyy_mm_dd_to_dd_mm_yyyy("2023-12-31"), "31-12-2023")
check(yyyy_mm_dd_to_dd_mm_yyyy("Today is 2024-01-05."), "Today is 05-01-2024.")  # 只改日期部分

PASS: '15-03-2024'
PASS: '31-12-2023'
PASS: 'Today is 05-01-2024.'


True

### 1.8 找出给定字符串中所有以 a 或 e 开头的单词

`\b` 保证从单词边界开始，`[aeAE]` 匹配大小写 a/e，`\w*` 匹配单词剩余部分。

In [10]:
def find_words_starting_with_a_or_e(s):
    """找出所有以 a 或 e 开头的单词（大小写都匹配）。"""
    return re.findall(r"\b[aeAE]\w*", s)


check(find_words_starting_with_a_or_e("An apple and an eagle"), ["An", "apple", "and", "an", "eagle"])
check(find_words_starting_with_a_or_e("Early birds eat ants"), ["Early", "eat", "ants"])
check(find_words_starting_with_a_or_e("big dog"), [])

PASS: ['An', 'apple', 'and', 'an', 'eagle']
PASS: ['Early', 'eat', 'ants']
PASS: []


True

### 1.9 将字符串中的 Road 缩写为 Rd

`\bRoad\b` 用单词边界保证只替换独立的单词 Road，不影响 Roadside、Broadway 等。

In [11]:
def abbreviate_road(s):
    """把独立的单词 Road 缩写为 Rd。"""
    return re.sub(r"\bRoad\b", "Rd", s)


check(abbreviate_road("100 Road Street"), "100 Rd Street")
check(abbreviate_road("Old Road and Cross Road"), "Old Rd and Cross Rd")
check(abbreviate_road("Broadway"), "Broadway")          # 单词内的 road 不替换
check(abbreviate_road("Roadside Road"), "Roadside Rd")  # Roadside 不替换

PASS: '100 Rd Street'
PASS: 'Old Rd and Cross Rd'
PASS: 'Broadway'
PASS: 'Roadside Rd'


True

### 1.10 找出字符串中所有长度至少为 4 个字符的单词

`\w{4,}` 表示 4 个及以上单词字符（字母、数字、下划线），两侧加 `\b` 防止截取长单词的一部分。

In [12]:
def find_words_at_least_4_chars(s):
    """找出所有长度 >= 4 的单词。"""
    return re.findall(r"\b\w{4,}\b", s)


check(find_words_at_least_4_chars("I have many words here"), ["have", "many", "words", "here"])
check(find_words_at_least_4_chars("one two three four five"), ["three", "four", "five"])
check(find_words_at_least_4_chars("a ab abc"), [])

PASS: ['have', 'many', 'words', 'here']
PASS: ['three', 'four', 'five']
PASS: []


True

### 1.11 删除字符串中除字母和数字之外的所有字符

`[^A-Za-z0-9]+` 匹配一段连续的非字母数字字符，替换为空串；`+` 一次删掉整段，效率更高。

In [13]:
def remove_non_alnum(s):
    """删除字符串中除字母和数字之外的所有字符。"""
    return re.sub(r"[^A-Za-z0-9]+", "", s)


check(remove_non_alnum("Hello, world! 123"), "Helloworld123")
check(remove_non_alnum("a@b#c$d%1"), "abcd1")
check(remove_non_alnum("你好 python_3"), "python3")

PASS: 'Helloworld123'
PASS: 'abcd1'
PASS: 'python3'


True

### 1.12 删除字符串中的括号及括号内的内容

`\([^()]*\)` 匹配不含嵌套括号的一对圆括号；前面加 `\s*` 一并去掉括号前的空白，使结果不带多余空格。按题目要求逐项处理四条输入。

In [14]:
def remove_parentheses_content(s):
    """删除括号及括号内的内容（含括号前的空白）；不支持嵌套括号。"""
    return re.sub(r"\s*\([^()]*\)", "", s)


check(remove_parentheses_content("example (.com)"), "example")
check(remove_parentheses_content("w3resource"), "w3resource")
check(remove_parentheses_content("github (.com)"), "github")
check(remove_parentheses_content("stackoverflow (.com)"), "stackoverflow")
check(remove_parentheses_content("abc (123) def"), "abc def")

PASS: 'example'
PASS: 'w3resource'
PASS: 'github'
PASS: 'stackoverflow'
PASS: 'abc def'


True

## 2 从网页检索气候数据

本部分用 `BeautifulSoup` 从城市页面中查找气候数据表格。参考文档：[BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)、[Requests](https://requests.readthedocs.io/)。

### 2.0 网络环境说明与页面选择

作业建议的维基百科页面（Madison / Buenos Aires / Harbin）在本机网络环境下**无法访问**（直连被防火墙阻断）：

| 尝试的地址 | 结果 |
| --- | --- |
| `https://en.wikipedia.org/wiki/...` | `ConnectTimeout` / `ReadTimeout`（连接超时，见下方实测输出） |
| `https://zh.wikipedia.org/wiki/...` | `ReadTimeout` |
| `https://baike.baidu.com/item/...` | HTTP 403（安全验证页） |
| 多个维基镜像站（wikimili / wanweibaike / baiwanzhan 等） | 连接失败或反爬验证 |

依据作业说明“若访问返回 403、429 或其他错误，请说明错误，并更换**可正常访问且包含气候数据表格**的页面”，改用以下官方气象页面（访问日期：**2026-11-26**）：

1. **日本气象厅（JMA）气候平年值页面**（东京、札幌）：
   `https://www.data.jma.go.jp/obd/stats/etrn/view/nml_sfc_ym.php?prec_no=44&block_no=47662&year=&month=&day=&view=`
2. **香港天文台（HKO）1991-2020 气候平均值页面**（结构差异测试）：
   `https://www.hko.gov.hk/tc/cis/normal/1991_2020/normals.htm`

两个站点表格结构有明显差异（详见 2.1 的观察结论），可用于检验函数对结构差异的鲁棒性。

In [15]:
import requests

ACCESS_DATE = "2026-11-26"   # 访问日期

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8,ja;q=0.7",
}
TIMEOUT = 15

# 演示：作业指定的维基百科页面在本网络下连接超时
try:
    requests.get("https://en.wikipedia.org/wiki/Madison,_Wisconsin",
                 headers=HEADERS, timeout=8)
except Exception as e:
    print("en.wikipedia.org ->", type(e).__name__, ":", e)

# 实际使用的页面 URL（均含气候数据表格）
TOKYO_URL = ("https://www.data.jma.go.jp/obd/stats/etrn/view/nml_sfc_ym.php"
             "?prec_no=44&block_no=47662&year=&month=&day=&view=")
SAPPORO_URL = ("https://www.data.jma.go.jp/obd/stats/etrn/view/nml_sfc_ym.php"
               "?prec_no=14&block_no=47412&year=&month=&day=&view=")
HKO_URL = "https://www.hko.gov.hk/tc/cis/normal/1991_2020/normals.htm"
NO_TABLE_URL = "https://www.weather.com.cn/cityintro/101050101.shtml"  # 不含表格的页面

print("URL 常量已定义，访问日期:", ACCESS_DATE)

en.wikipedia.org -> ReadTimeout : HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read timeout=8)
URL 常量已定义，访问日期: 2026-11-26


### 2.1 问题 1：读取 HTML、解析并定位气候数据表格，观察其结构

读取东京（JMA）页面，用标准解析器 `html.parser` 解析，定位气候数据表格（`<table id="tablefix1">`），观察其直接子节点与行列结构，并与 HKO 表格对比。

In [16]:
from bs4 import BeautifulSoup

# ---- 读取并解析东京(JMA)页面 ----
resp = requests.get(TOKYO_URL, headers=HEADERS, timeout=TIMEOUT)
if not resp.encoding or resp.encoding.lower() in ("iso-8859-1", "latin-1"):
    resp.encoding = resp.apparent_encoding          # 无 charset 时按内容推断编码
soup = BeautifulSoup(resp.text, "html.parser")
print("页面 <title>:", soup.title.get_text(strip=True))
print("页面 <table> 总数:", len(soup.find_all("table")))

# ---- 定位气候数据表格 ----
table = soup.find("table", id="tablefix1")
print("表格属性: id =", table.get("id"), "| class =", table.get("class"))

direct = [c.name for c in table.children if c.name]
print("表格直接子节点:", direct, "-> 共", len(direct), "个")

rows = table.find_all("tr")
print("总行数:", len(rows))
for i in (0, 1, 3, 5):
    cells = [(c.name, c.get("scope"), " ".join(c.get_text().split())[:14])
             for c in rows[i].find_all(["th", "td"], recursive=False)]
    print(f"第 {i} 行(前 6 格):", cells[:6])
print("...")
print("表头 th 的 scope 取值:", sorted({t.get("scope") for t in table.find_all("th")}))
print("数据行行标签 th[scope=row] 前 5 个:",
      [t.get_text(strip=True) for t in table.find_all("th", scope="row")][:5])

# ---- 对比:HKO 页面的表格结构 ----
hko_resp = requests.get(HKO_URL, headers=HEADERS, timeout=TIMEOUT)
if not hko_resp.encoding or hko_resp.encoding.lower() in ("iso-8859-1", "latin-1"):
    hko_resp.encoding = hko_resp.apparent_encoding
hko_soup = BeautifulSoup(hko_resp.text, "html.parser")
hko_table = hko_soup.find_all("table")[1]           # 表2(降雨量)
print("HKO 表2 直接子节点:", [c.name for c in hko_table.children if c.name])
print("HKO 表2 行数:", len(hko_table.find_all("tr")))
hko_row = hko_table.find_all("tr")[2]
print("HKO 表2 第 2 行单元格:", [(c.name, " ".join(c.get_text().split())[:10])
                                for c in hko_row.find_all(["th", "td"], recursive=False)])

页面 <title>: 気象庁｜過去の気象データ検索
页面 <table> 总数: 6
表格属性: id = tablefix1 | class = ['data2_s']
表格直接子节点: ['tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr', 'tr'] -> 共 18 个
总行数: 18
第 0 行(前 6 格): [('th', 'row', '要素'), ('th', 'colgroup', '気圧'), ('th', 'colgroup', '降水量'), ('th', 'colgroup', '気温'), ('th', 'colgroup', '蒸気圧'), ('th', 'colgroup', '相対湿度')]
第 1 行(前 6 格): [('th', 'col', '現地平均(hPa)'), ('th', 'col', '海面平均(hPa)'), ('th', 'col', '合計(mm)'), ('th', 'col', '平均(℃)'), ('th', 'col', '日最高(℃)'), ('th', 'col', '日最低(℃)')]
第 3 行(前 6 格): [('th', 'row', '統計期間'), ('td', None, '1991～2020'), ('td', None, '1991～2020'), ('td', None, '1991～2020'), ('td', None, '1991～2020'), ('td', None, '1991～2020')]
第 5 行(前 6 格): [('th', 'row', '1月'), ('td', None, '1012.8'), ('td', None, '1015.8'), ('td', None, '59.7'), ('td', None, '5.4'), ('td', None, '9.8')]
...
表头 th 的 scope 取值: ['col', 'colgroup', 'row']
数据行行标签 th[scope=row] 前 5 个: ['要素', '統計期間', '資料年数', '1月', '2月']
HKO 

**观察结论**

- **JMA 表格**：直接子节点是 **18 个 `<tr>`**（没有 `<caption>`/`<tbody>` 包装）。第 0~2 行是列标题（`<th scope="colgroup">` 元素分组、`<th scope="col">` 子要素）；第 3、4 行是 `統計期間`、`資料年数`；第 5~17 行是数据行，**行标签（1月~12月、年）位于 `<th scope="row">`**。
- **HKO 表格**：直接子节点是 **`<caption>` + `<tbody>`**；标题“表 2. …”放在 `<caption>` 中；列标题行整行都是 `<th>`；而**行标签（一月~十二月、年值）用的是普通 `<td>`**，与 JMA 用 `<th scope="row">` 不同。
- 作业描述的 Wikipedia 结构（“Climate data for” 标题行 + 月份列标题 + `scope="row"` 行名 + 数据来源行）与上述两个站点都不同，因此下面两个函数把三种结构中**遇到的常见情况都做了处理**。

### 2.2 问题 2：`retrieve_climate_table(s)`

只接收一个 URL 字符串：
- 页面中有气候数据表格 → 返回对应的 BeautifulSoup `Tag`；
- 成功读取但没有此类表格 → 返回 `None`；
- 网络错误或 HTTP 错误 → **抛出异常**。

识别策略：先找标题含 “Climate data for” 的表格（Wikipedia 风格）；否则按“文本含气候要素关键词 + 出现至少两种不同月份”筛选；有多个候选时返回第一个（作业允许返回其中任意一个）；一个都没有则返回 `None`。

In [17]:
import re

from bs4 import BeautifulSoup, Tag

CLIMATE_TITLE_RE = re.compile(r"climate data for", re.IGNORECASE)
CLIMATE_KEYWORD_RE = re.compile(
    r"climate|weather|temperature|precipitation|rain|rainfall|humidity|sunshine|snow|cloud"
    r"|気温|气温|氣溫|温度|溫度|降水|降雨|湿度|濕度|日照|降雪|積雪|雲量|云量|気象|氣象",
    re.IGNORECASE,
)
MONTH_RE = re.compile(
    r"[1-9]月|1[0-2]月|一月|二月|三月|四月|五月|六月|七月|八月|九月|十月|十一月|十二月"
    r"|Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?"
    r"|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?",
    re.IGNORECASE,
)


def _norm_text(node):
    """合并连续空白字符并去除首尾空白。"""
    return " ".join(node.get_text().split())


def _fetch_and_parse(url):
    """GET url 并解析为 BeautifulSoup；网络错误 / HTTP 错误直接抛出异常。"""
    resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    resp.raise_for_status()                          # 4xx/5xx -> HTTPError
    if not resp.encoding or resp.encoding.lower() in ("iso-8859-1", "latin-1"):
        resp.encoding = resp.apparent_encoding       # 无 charset 时按内容推断
    return BeautifulSoup(resp.text, "html.parser")


def _find_climate_table(soup):
    """在已解析的页面中定位气候数据表格，返回 Tag 或 None。"""
    tables = soup.find_all("table")
    for table in tables:                             # 1) 标题优先(Wikipedia 风格)
        if CLIMATE_TITLE_RE.search(_norm_text(table)):
            return table
    candidates = []
    for table in tables:                             # 2) 关键词 + 月份启发式
        text = _norm_text(table)
        if CLIMATE_KEYWORD_RE.search(text) and len(set(MONTH_RE.findall(text))) >= 2:
            candidates.append(table)
    if not candidates:
        return None
    return candidates[0]                    # 多个候选时返回第一个(作业允许返回任意一个)


def retrieve_climate_table(s):
    """接收一个 URL 字符串。

    页面含气候数据表格 -> 返回对应的 BeautifulSoup Tag；
    成功读取但无此类表格 -> 返回 None；网络错误 / HTTP 错误 -> 抛出异常。
    """
    soup = _fetch_and_parse(s)
    return _find_climate_table(soup)


# ---- 测试 1:JMA 东京(有气候数据表格) ----
table = retrieve_climate_table(TOKYO_URL)
print("东京(JMA) ->", type(table).__name__, "| id:", table.get("id"), "| class:", table.get("class"))
print("表格文本前 80 字:", _norm_text(table)[:80], "...")

# ---- 测试 2:JMA 札幌(另一个城市页面,同一模板) ----
table = retrieve_climate_table(SAPPORO_URL)
print("札幌(JMA) ->", type(table).__name__, "| id:", table.get("id"), "| class:", table.get("class"))

# ---- 测试 3:HKO 香港(结构有差异的页面) ----
table = retrieve_climate_table(HKO_URL)
print("香港(HKO) ->", type(table).__name__, "| caption:", _norm_text(table.find("caption"))[:50])

# ---- 测试 4:成功读取但无气候数据表格 -> None ----
print("无气候表格页面 ->", retrieve_climate_table(NO_TABLE_URL))

# ---- 测试 5:网络错误 -> 抛异常 ----
try:
    retrieve_climate_table("https://this-domain-does-not-exist-xyz.invalid/")
except Exception as e:
    print("不可达域名 ->", type(e).__name__, ":", e)

# ---- 测试 6:HTTP 错误(404) -> 抛异常 ----
try:
    retrieve_climate_table("https://www.data.jma.go.jp/obd/stats/etrn/view/nonexistent.php")
except Exception as e:
    print("HTTP 404 页面 ->", type(e).__name__, ":", e)

ReadTimeout: HTTPSConnectionPool(host='www.data.jma.go.jp', port=443): Read timed out. (read timeout=15)

### 2.3 问题 3：`list_climate_table_row_names(s)`

只接收一个城市页面 URL：
- 找到气候数据表格 → 返回其**行名**组成的列表（每项都是去除首尾空白的纯 Python 字符串）；
- 成功读取但找不到 → 返回 `None`；网络/HTTP 错误同样抛出异常。

行名提取规则（排除项）：
1. 空行、只有**一个单元格**的行（整行标题行、数据来源行，如 Wikipedia 的 `Source:` 行）；
2. **整行都是 `<th>`** 的行（列标题行，如 Wikipedia 的月份表头、JMA 的要素表头）；
3. 行标签文本含 “Climate data for” / “表 N” / 统计期间 / 数据来源关键词的行；
4. 无 `<th>` 时取行首 `<td>` 作为行标签（HKO 结构），但标签含数字说明是纯数据行，跳过。

In [ ]:
SOURCE_PERIOD_RE = re.compile(
    r"統計期間|資料年数|統計期|資料來源|数据来源|來源|source|statistical period",
    re.IGNORECASE,
)


def _row_names_from_table(table):
    """从气候数据表格中提取行名列表(纯字符串,不含 Tag / HTML 标签)。"""
    names = []
    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"], recursive=False)  # 只取该行的直接单元格
        if not cells or len(cells) == 1:
            continue                                       # 空行 / 占整行的标题、来源行
        ths = [c for c in cells if c.name == "th"]
        if ths and len(ths) == len(cells):
            continue                                       # 整行都是 th:列标题行(月份表头等)
        if ths:
            label = next((c for c in ths if c.get("scope") == "row"), ths[0])
        else:
            label = cells[0]                               # 无 th 时取首单元格(HKO 结构)
        text = _norm_text(label)
        if not text:
            continue
        if CLIMATE_TITLE_RE.search(text) or re.match(r"^表\s*\d+", text):
            continue                                       # 表格标题行
        if SOURCE_PERIOD_RE.search(text):
            continue                                       # 统计期间 / 数据来源行
        if label.name == "td" and re.search(r"\d", text):
            continue                                       # td 含数字:纯数据行,不是行名
        names.append(text)
    return names


def list_climate_table_row_names(s):
    """接收一个城市页面 URL。

    找到气候数据表格 -> 返回行名列表(去首尾空白的纯字符串);
    成功读取但找不到 -> 返回 None;网络错误 / HTTP 错误 -> 抛出异常。
    """
    table = retrieve_climate_table(s)
    if table is None:
        return None
    return _row_names_from_table(table)


# ---- 测试 1:JMA 东京 ----
names = list_climate_table_row_names(TOKYO_URL)
print("东京:", names)
check(len(names), 13)
check(names[:12], [f"{m}月" for m in range(1, 13)])   # 月份是行标签,不含标题/表头行
check(names[-1], "年")
check(all(isinstance(n, str) and "<" not in n for n in names), True)

# ---- 测试 2:JMA 札幌 ----
names = list_climate_table_row_names(SAPPORO_URL)
print("札幌:", names)
check(len(names), 13)
check(names[:12], [f"{m}月" for m in range(1, 13)])

# ---- 测试 3:HKO 香港(结构有差异) ----
names = list_climate_table_row_names(HKO_URL)
print("香港:", names)
check(len(names), 13)
check(names[:12], ["一月", "二月", "三月", "四月", "五月", "六月",
                   "七月", "八月", "九月", "十月", "十一月", "十二月"])
check(all(isinstance(n, str) and "<" not in n for n in names), True)

# ---- 测试 4:无气候数据表格 -> None ----
print("无气候表格页面 ->", list_climate_table_row_names(NO_TABLE_URL))

# ---- 测试 5:离线验证 Wikipedia 风格表格(本网络无法访问维基百科,用等价 HTML 片段验证识别逻辑) ----
WIKI_STYLE_HTML = """
<html><body>
<table class="navbox"><tr><th>导航</th><td><a href="/wiki/Main_Page">Main Page</a></td></tr></table>
<table class="wikitable mw-collapsible">
<tbody>
<tr><th colspan="14">Climate data for Harbin (1991&#8211;2020 normals)</th></tr>
<tr><th scope="col">Month</th><th scope="col">Jan</th><th scope="col">Feb</th><th scope="col">Mar</th>
<th scope="col">Apr</th><th scope="col">May</th><th scope="col">Jun</th><th scope="col">Jul</th>
<th scope="col">Aug</th><th scope="col">Sep</th><th scope="col">Oct</th><th scope="col">Nov</th>
<th scope="col">Dec</th><th scope="col">Year</th></tr>
<tr><th scope="row">Record high °C (°F)</th><td>4</td><td>9</td><td>21</td><td>30</td><td>34</td><td>38</td><td>36</td><td>36</td><td>31</td><td>27</td><td>17</td><td>8</td><td>38</td></tr>
<tr><th scope="row">Average high °C (°F)</th><td>&#8722;12.3</td><td>&#8722;6.5</td><td>2.8</td><td>13.9</td><td>21.3</td><td>26.0</td><td>27.8</td><td>26.5</td><td>21.2</td><td>12.4</td><td>&#8722;0.1</td><td>&#8722;9.5</td><td>10.3</td></tr>
<tr><th scope="row">Daily mean °C (°F)</th><td>&#8722;17.6</td><td>&#8722;11.9</td><td>&#8722;2.7</td><td>8.0</td><td>15.3</td><td>20.6</td><td>23.1</td><td>21.6</td><td>15.1</td><td>6.4</td><td>&#8722;5.2</td><td>&#8722;14.8</td><td>4.9</td></tr>
<tr><th scope="row">Average low °C (°F)</th><td>&#8722;22.9</td><td>&#8722;17.2</td><td>&#8722;8.2</td><td>2.1</td><td>9.2</td><td>15.3</td><td>18.5</td><td>16.9</td><td>9.4</td><td>0.9</td><td>&#8722;10.1</td><td>&#8722;19.9</td><td>&#8722;0.6</td></tr>
<tr><th scope="row">Average precipitation mm (inches)</th><td>4.4</td><td>5.4</td><td>11.1</td><td>23.9</td><td>41.4</td><td>91.4</td><td>142.6</td><td>121.6</td><td>58.4</td><td>25.1</td><td>13.8</td><td>8.3</td><td>547.4</td></tr>
<tr><td colspan="14">Source: China Meteorological Administration</td></tr>
</tbody>
</table>
</body></html>
"""
wiki_soup = BeautifulSoup(WIKI_STYLE_HTML, "html.parser")
wiki_table = _find_climate_table(wiki_soup)
print("识别到的表格:", wiki_table.get("class"), "| 标题行:", _norm_text(wiki_table.find("tr")))
wiki_names = _row_names_from_table(wiki_table)
print("行名:", wiki_names)
check(wiki_names, ["Record high °C (°F)", "Average high °C (°F)", "Daily mean °C (°F)",
                   "Average low °C (°F)", "Average precipitation mm (inches)"])

东京: ['1月', '2月', '3月', '4月', '5月', '6月', '7月', '8月', '9月', '10月', '11月', '12月', '年']
PASS: 13
PASS: ['1月', '2月', '3月', '4月', '5月', '6月', '7月', '8月', '9月', '10月', '11月', '12月']
PASS: '年'
PASS: True


札幌: ['1月', '2月', '3月', '4月', '5月', '6月', '7月', '8月', '9月', '10月', '11月', '12月', '年']
PASS: 13
PASS: ['1月', '2月', '3月', '4月', '5月', '6月', '7月', '8月', '9月', '10月', '11月', '12月']


香港: ['一月', '二月', '三月', '四月', '五月', '六月', '七月', '八月', '九月', '十月', '十一月', '十二月', '年值']
PASS: 13
PASS: ['一月', '二月', '三月', '四月', '五月', '六月', '七月', '八月', '九月', '十月', '十一月', '十二月']
PASS: True
无气候表格页面 -> None
识别到的表格: ['wikitable', 'mw-collapsible'] | 标题行: Climate data for Harbin (1991–2020 normals)
行名: ['Record high °C (°F)', 'Average high °C (°F)', 'Daily mean °C (°F)', 'Average low °C (°F)', 'Average precipitation mm (inches)']
PASS: ['Record high °C (°F)', 'Average high °C (°F)', 'Daily mean °C (°F)', 'Average low °C (°F)', 'Average precipitation mm (inches)']


True